# Notebook 05: Context Management Pattern

**CCA Pattern:** Structured Summaries vs Raw Transcript

This notebook demonstrates the lost-in-middle effect and the correct remedy:

1. **Anti-Pattern**: Raw transcript accumulator — O(n) growth, early facts buried
2. **Correct Pattern**: `ContextSummary` — bounded size, key facts always preserved
3. **Compare**: Token estimate after 5 turns — raw vs structured

## Setup

Install dependencies and set up services. Requires `ANTHROPIC_API_KEY` environment variable.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path(".").resolve()))

import anthropic
from helpers import compare_results

from customer_service.agent.agent_loop import run_agent_loop
from customer_service.agent.context_manager import TOKEN_BUDGET, ContextSummary
from customer_service.agent.system_prompts import get_system_prompt
from customer_service.anti_patterns.raw_transcript import RawTranscriptContext
from customer_service.services.audit_log import AuditLog
from customer_service.services.container import ServiceContainer
from customer_service.services.customer_db import CustomerDatabase
from customer_service.services.escalation_queue import EscalationQueue
from customer_service.services.financial_system import FinancialSystem
from customer_service.services.policy_engine import PolicyEngine

In [ ]:
def make_services():
    return ServiceContainer(
        customer_db=CustomerDatabase(),
        policy_engine=PolicyEngine(),
        financial_system=FinancialSystem(),
        escalation_queue=EscalationQueue(),
        audit_log=AuditLog(),
    )


client = anthropic.Anthropic()

## The Problem: Lost-in-Middle Effect

When agents accumulate raw conversation transcripts as context:

- Each turn adds the full user message + assistant response + tool results
- Token count grows **O(n)** — proportional to number of turns
- Critical early facts get **buried** under later content
- The model's attention focuses on recent content, **ignoring early context**

> **CCA Exam fact:** A bigger context window makes the lost-in-middle effect > **WORSE**, not better. More context = more dilution, not less.

We will simulate a 5-turn conversation where the customer mentions an important fact in Turn 1 ("birthday gift, deadline March 30th"). By Turn 5, this fact is buried in the raw transcript but remains visible in the structured summary.

## Anti-Pattern: Raw Transcript Accumulator

<div style="background-color:#fff0f0; border-left:4px solid #cc0000; padding:16px; margin:12px 0; border-radius:4px;">

**WRONG APPROACH: Append every message to a raw text string**

The `RawTranscriptContext` appends every user message and assistant response as raw text. The token count grows without bound, and early facts get lost.

</div>

In [ ]:
raw_transcript = RawTranscriptContext()

# 5-turn conversation for C001 (Alice Johnson, Regular tier)
# IMPORTANT EARLY FACT in message 1: birthday gift, March 30 deadline
# Messages 2-5 are routine follow-ups — the early fact gets buried
user_messages = [
    # Turn 1: Important early fact — birthday gift, hard deadline
    "I need help with order #ORD-001. It was a birthday gift for my mother "
    "and the item arrived broken. Her birthday is March 30th — I really need "
    "this resolved before then. My customer ID is C001.",
    # Turn 2: Follow-up asking about process
    "What is the refund policy for defective items? Will I get a full refund or just store credit?",
    # Turn 3: Providing additional details
    "I can confirm the item was defective right out of the box — it would not "
    "turn on at all. I have photos of the damage if needed.",
    # Turn 4: Asking about timeline
    "How long does the refund take to process? Will it show up on my card within a few days?",
    # Turn 5: Final confirmation request
    "Can you confirm that the refund has been processed and summarize "
    "what was resolved in this session?",
]

print(f"Conversation: {len(user_messages)} turns")
print("Customer: C001 (Alice Johnson, Regular tier)")
print("Key early fact: birthday gift, March 30th deadline (Turn 1)")

### Anti-Pattern Turn 1 — Important early fact introduced

The customer mentions the **birthday gift** and the **March 30th deadline**. This critical context appears in the transcript now — but watch what happens by Turn 5.

In [ ]:
# Turn 1 — anti-pattern
msg1 = user_messages[0]
raw_transcript.append("user", msg1)
result1_anti = run_agent_loop(client, make_services(), msg1, get_system_prompt())
raw_transcript.append("assistant", result1_anti.final_response or "(no text response)")

tokens_after_t1 = raw_transcript.token_estimate()
context_t1 = raw_transcript.to_context_string()
print(f"Tokens after Turn 1: {tokens_after_t1}")
print(f"'birthday' in last 200 chars: {'birthday' in context_t1[-200:]}")
print(f"'birthday' in full transcript: {'birthday' in context_t1}")

### Anti-Pattern Turn 2 — Routine follow-up

In [ ]:
# Turn 2 — anti-pattern
msg2 = user_messages[1]
raw_transcript.append("user", msg2)
result2_anti = run_agent_loop(client, make_services(), msg2, get_system_prompt())
raw_transcript.append("assistant", result2_anti.final_response or "(no text response)")

tokens_after_t2 = raw_transcript.token_estimate()
context_t2 = raw_transcript.to_context_string()
delta_t2 = tokens_after_t2 - tokens_after_t1
print(f"Tokens after Turn 2: {tokens_after_t2} (was {tokens_after_t1} — +{delta_t2})")
print(f"'birthday' in last 200 chars: {'birthday' in context_t2[-200:]}")

### Anti-Pattern Turn 3 — More details added

In [ ]:
# Turn 3 — anti-pattern
msg3 = user_messages[2]
raw_transcript.append("user", msg3)
result3_anti = run_agent_loop(client, make_services(), msg3, get_system_prompt())
raw_transcript.append("assistant", result3_anti.final_response or "(no text response)")

tokens_after_t3 = raw_transcript.token_estimate()
context_t3 = raw_transcript.to_context_string()
delta_t3 = tokens_after_t3 - tokens_after_t2
print(f"Tokens after Turn 3: {tokens_after_t3} (was {tokens_after_t2} — +{delta_t3})")
print(f"'birthday' in last 200 chars: {'birthday' in context_t3[-200:]}")

### Anti-Pattern Turn 4 — Transcript keeps growing

In [ ]:
# Turn 4 — anti-pattern
msg4 = user_messages[3]
raw_transcript.append("user", msg4)
result4_anti = run_agent_loop(client, make_services(), msg4, get_system_prompt())
raw_transcript.append("assistant", result4_anti.final_response or "(no text response)")

tokens_after_t4 = raw_transcript.token_estimate()
context_t4 = raw_transcript.to_context_string()
delta_t4 = tokens_after_t4 - tokens_after_t3
print(f"Tokens after Turn 4: {tokens_after_t4} (was {tokens_after_t3} — +{delta_t4})")
print(f"'birthday' in last 200 chars: {'birthday' in context_t4[-200:]}")

### Anti-Pattern Turn 5 — Early fact is now buried

By Turn 5, the birthday/deadline fact from Turn 1 has been pushed out of the last 200 characters of the transcript. The model's attention is on recent content — the early critical fact is effectively **lost in the middle**.

In [ ]:
# Turn 5 — anti-pattern: early fact now buried
msg5 = user_messages[4]
raw_transcript.append("user", msg5)
result5_anti = run_agent_loop(client, make_services(), msg5, get_system_prompt())
raw_transcript.append("assistant", result5_anti.final_response or "(no text response)")

tokens_after_t5 = raw_transcript.token_estimate()
context_t5 = raw_transcript.to_context_string()
delta_t5 = tokens_after_t5 - tokens_after_t4
print(f"Tokens after Turn 5: {tokens_after_t5} (was {tokens_after_t4} — +{delta_t5})")
print(f"'birthday' in last 200 chars: {'birthday' in context_t5[-200:]}")
print(f"'birthday' in full transcript: {'birthday' in context_t5}")
print()
growth = (
    f"{tokens_after_t1} -> {tokens_after_t2} -> "
    f"{tokens_after_t3} -> {tokens_after_t4} -> {tokens_after_t5}"
)
print(f"Token growth: {growth}")
print("Pattern: O(n) — tokens grow linearly with each turn")

## Correct Pattern: Structured Context Summary

<div style="background-color:#f0fff0; border-left:4px solid #00aa00; padding:16px; margin:12px 0; border-radius:4px;">

**CORRECT APPROACH: `ContextSummary` with fixed-field schema**

The `ContextSummary` dataclass maintains structured fields: `customer_id`, `issue_type`, `tools_called`, `decisions_made`, `pending_actions`. Key facts are stored in named fields — they cannot be buried or compacted away.

</div>

We run the same 5-turn conversation but now use `ContextSummary.update()` after each turn. Watch:

- `token_estimate` stays bounded (never exceeds `TOKEN_BUDGET`)
- `customer_id` and `issue_type` survive every compaction
- Structured fields preserve key facts regardless of turn count

In [ ]:
summary = ContextSummary()
summary.customer_id = "C001"
summary.issue_type = "defective_item_refund"

# Note the birthday fact is captured in a pending_action — a structured field
summary.pending_actions.append("birthday gift deadline: March 30th")

print(f"TOKEN_BUDGET: {TOKEN_BUDGET} chars (~{TOKEN_BUDGET // 4} tokens)")
print(f"Initial token_estimate: {summary.token_estimate} (stale until first update())")
print()
print("Initial structured context:")
print(summary.to_system_context())

### Correct Pattern Turn 1 — Key fact captured in structured field

In [ ]:
# Turn 1 — correct pattern
result1_correct = run_agent_loop(client, make_services(), user_messages[0], get_system_prompt())
summary.update("lookup_customer", "C001 Alice Johnson, Regular tier, no holds")

print(f"token_estimate after Turn 1: {summary.token_estimate}")
print(f"customer_id: {summary.customer_id}")
print(f"'birthday' in pending_actions: {'birthday' in str(summary.pending_actions)}")
print()
print(summary.to_system_context())

### Correct Pattern Turn 2 — Policy check recorded

In [ ]:
# Turn 2 — correct pattern
result2_correct = run_agent_loop(client, make_services(), user_messages[1], get_system_prompt())
summary.update("check_policy", "Defective item: full refund eligible, Standard tier")

print(f"token_estimate after Turn 2: {summary.token_estimate}")
print(f"'birthday' in pending_actions: {'birthday' in str(summary.pending_actions)}")
print()
print(summary.to_system_context())

### Correct Pattern Turn 3 — Photo evidence noted

In [ ]:
# Turn 3 — correct pattern
result3_correct = run_agent_loop(client, make_services(), user_messages[2], get_system_prompt())
summary.update("check_policy", "Photo evidence provided, defect confirmed")

print(f"token_estimate after Turn 3: {summary.token_estimate}")
print(f"'birthday' in pending_actions: {'birthday' in str(summary.pending_actions)}")
print()
print(summary.to_system_context())

### Correct Pattern Turn 4 — Refund processed

In [ ]:
# Turn 4 — correct pattern
result4_correct = run_agent_loop(client, make_services(), user_messages[3], get_system_prompt())
summary.update("process_refund", "Refund approved, 3-5 business days to card")

print(f"token_estimate after Turn 4: {summary.token_estimate}")
print(f"'birthday' in pending_actions: {'birthday' in str(summary.pending_actions)}")
print()
print(summary.to_system_context())

### Correct Pattern Turn 5 — Key fact still visible

After 5 turns, the birthday/deadline fact is **still visible** in `pending_actions`. The token estimate is bounded — compaction may have fired, but structured fields survive. The early critical fact is never lost.

In [ ]:
# Turn 5 — correct pattern: early fact still preserved
result5_correct = run_agent_loop(client, make_services(), user_messages[4], get_system_prompt())
summary.update("log_interaction", "Session complete: refund processed, audit logged")

print(f"token_estimate after Turn 5: {summary.token_estimate}")
print(f"(TOKEN_BUDGET is {TOKEN_BUDGET})")
print(f"Under budget: {summary.token_estimate <= TOKEN_BUDGET}")
print(f"'birthday' in pending_actions: {'birthday' in str(summary.pending_actions)}")
print(f"'birthday' in full context: {'birthday' in summary.to_system_context()}")
print()
print("Final structured context:")
print(summary.to_system_context())

## Compare Results

Direct comparison after 5 turns: raw transcript vs structured summary.

In [ ]:
compare_results(
    {
        "final_token_estimate": raw_transcript.token_estimate(),
        "birthday_fact_visible": "birthday" in raw_transcript.to_context_string()[-200:],
        "growth_pattern": "O(n) — unbounded",
    },
    {
        "final_token_estimate": summary.token_estimate,
        "birthday_fact_visible": "birthday" in summary.to_system_context(),
        "growth_pattern": f"O(1) — bounded at {TOKEN_BUDGET}",
    },
)

## CCA Exam Tip: Context Management

> **CCA Exam Tip:**
> 
> **The Lost-in-Middle Effect:**
> - Raw transcripts cause O(n) token growth — each turn makes the context longer
> - Important facts from early turns get buried under later content
> - The model's attention focuses on recent content; early facts are diluted
> - **Bigger context window makes it WORSE, not better** — more dilution, not less
> 
> **Correct remedy:** `ContextSummary` with fixed-field schema:
> - `customer_id`, `issue_type` — structured fields that survive compaction
> - `decisions_made`, `pending_actions` — short bullets, bounded by `TOKEN_BUDGET`
> - `token_estimate` — stays bounded regardless of turn count
> 
> **Exam signal → Answer:**
> - "Quality drops in longer sessions" → Context degradation → use structured summaries
> - "Important facts forgotten by turn 5" → Lost-in-middle → use `ContextSummary`
> - "Use a bigger context window" → TRAP — makes it worse

## Summary

| Metric | Anti-Pattern (Raw) | Correct (ContextSummary) |
|--------|-------------------|-------------------------|
| Token growth | O(n) — unbounded | O(1) — bounded at TOKEN_BUDGET |
| Early facts | Buried by turn 5 | Always in structured fields |
| Compaction | None — grows forever | Automatic, structured fields survive |
| API cost | Grows each turn | Fixed (bounded context injection) |

**Key files:**

- `src/customer_service/agent/context_manager.py` — `ContextSummary`, `TOKEN_BUDGET`
- `src/customer_service/anti_patterns/raw_transcript.py` — `RawTranscriptContext`

**CCA Rule:** 'Quality drops in longer sessions? → Context degradation → Use `ContextSummary` with fixed-field schema, not raw transcripts.'